In [2]:
import pandas as pd
df = pd.read_pickle('../output/df_bereinigt.pkl')
print("Geladen:", df.shape)
import re
import spacy

nlp = spacy.load('de_core_news_sm')

def clean_text(text):
    """Grundlegende Textbereinigung für deutsche Beschwerdetexte"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zäöüß\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_text'] = df['text_raw'].apply(clean_text)

print("Beispiel VORHER:", df['text_raw'].iloc[0][:200])
print("Beispiel NACHHER:", df['clean_text'].iloc[0][:200])

Geladen: (2541, 10)
Beispiel VORHER: Seit der Nacht von Sonntag auf Montag steht im Frieda-Sigrist-Weg 15 immer noch ein grosses Holzregal mit dem Hinweis "zum Verschenken", obwohl es schon mehrfach geregnet hat. Das Regal wird sich durc
Beispiel NACHHER: seit der nacht von sonntag auf montag steht im frieda sigrist weg immer noch ein grosses holzregal mit dem hinweis zum verschenken obwohl es schon mehrfach geregnet hat das regal wird sich durch das w


In [ ]:
!python -m spacy download de_core_news_sm

In [3]:
def advanced_preprocessing(text):
    """Lemmatisierung und Stoppwortentfernung mit dem deutschen spaCy-Modell"""
    if not text or len(text) < 3:
        return []
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token.text) > 2
        and token.pos_ not in ['PRON']
    ]
    return tokens

print("Starte Vorverarbeitung, das kann einige Minuten dauern...")
df['processed_tokens'] = df['clean_text'].apply(advanced_preprocessing)
df = df[df['processed_tokens'].apply(len) > 0].reset_index(drop=True)
print(f"Verbleibende Dokumente nach Vorverarbeitung: {len(df)}")

Starte Vorverarbeitung, das kann einige Minuten dauern...
Verbleibende Dokumente nach Vorverarbeitung: 2540


In [4]:
df.to_pickle('../output/df_preprocessed.pkl')
print("Gespeichert:", df.shape)

Gespeichert: (2540, 12)
